# LANL Cyber 1: prepare an ML-ready feature dataset

This notebook builds a broader ordinary-activity feature dataset with bounded streaming. It prepares chronological partitions for a future Isolation Forest experiment, but deliberately does not train a model.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import FEATURE_COLUMNS, chronological_split

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_train.parquet'

features = pd.read_parquet(FEATURE_PATH)
print('Loaded broader feature dataset:', FEATURE_PATH)
print('The red-team file is intentionally not used for feature generation or validation here.')


The full auth archive is not loaded. The next cell scans only through END_TIMESTAMP and keeps rolling state.


## Dataset summary

Inspect the generated artifact before any model is trained.

## Feature and context schema

Context columns identify the prediction row and are not model inputs. The 11 behavioral columns remain the only candidate ML features.

In [ ]:
context_columns = ['timestamp', 'entity', 'entity_type', 'window_start', 'window_end']
print('Shape:', features.shape)
print('Rows:', len(features))
print('Unique entities:', features['entity'].nunique())
print('Unique timestamps:', features['timestamp'].nunique())
print('Timestamp range:', int(features['timestamp'].min()), 'to', int(features['timestamp'].max()))
print('Rows per entity:')
display(features['entity'].value_counts().describe().to_frame('value'))
print('Feature columns:', FEATURE_COLUMNS)
print('Dtypes:')
display(features[context_columns + FEATURE_COLUMNS].dtypes.rename('dtype').to_frame())
display(features[FEATURE_COLUMNS].describe().T)


In [ ]:
numeric_features = features[FEATURE_COLUMNS]
missing_values = int(features[context_columns + FEATURE_COLUMNS].isna().sum().sum())
constant_features = [column for column in FEATURE_COLUMNS if numeric_features[column].nunique(dropna=False) <= 1]
correlation = numeric_features.corr().abs()
highly_redundant = sorted(
    (left, right, float(correlation.loc[left, right]))
    for index, left in enumerate(FEATURE_COLUMNS)
    for right in FEATURE_COLUMNS[index + 1:]
    if correlation.loc[left, right] >= 0.99
)

print('Missing values:', missing_values)
print('Duplicate rows:', int(features.duplicated().sum()))
print('Duplicate entity/timestamp keys:', int(features.duplicated(['entity', 'timestamp']).sum()))
print('Failed-auth rows:', int((features['failed_auth_count'] > 0).sum()))
print('Failed-auth events:', int(features['failed_auth_count'].sum()))
print('Failed-auth distribution:')
display(features['failed_auth_count'].value_counts().sort_index().rename('rows').to_frame())
print('Constant features:', constant_features)
print('Highly redundant features (absolute correlation >= 0.99):', highly_redundant)
print('Feature distributions:')
display(numeric_features.describe().T)


## Causal and chronological validation

Rows are split by timestamp groups. No labels, identifiers, or timestamps are included in the behavioral matrix.

In [ ]:
assert features['window_end'].eq(features['timestamp']).all()
assert not (features['window_end'] > features['timestamp']).any()
assert not (numeric_features < 0).any().any()
print('Temporal causality checks: passed')

train, validation, test = chronological_split(features)
splits = {'training': train, 'validation': validation, 'test': test}
for name, frame in splits.items():
    print(f'{name}: rows={len(frame)}, entities={frame.entity.nunique()}, timestamps={frame.timestamp.nunique()}, range={int(frame.timestamp.min())}..{int(frame.timestamp.max())}')
assert train.timestamp.max() < validation.timestamp.min() < test.timestamp.min()
print('Chronological split checks: passed')


In [ ]:
print('Feature distributions by chronological split:')
for name, frame in splits.items():
    print(f'--- {name} ---')
    display(frame[FEATURE_COLUMNS].describe().T[['mean', 'std', 'min', 'max']])


## Prepare feature matrices

Only numeric behavioral features are selected. This notebook stops before detector training.

## Feature matrices only

Scaling is not required by Isolation Forest. The matrices are prepared for a later notebook, but no model is imported or trained here.

In [ ]:
X_train = train[FEATURE_COLUMNS].copy()
X_validation = validation[FEATURE_COLUMNS].copy()
X_test = test[FEATURE_COLUMNS].copy()
print('X_train.shape:', X_train.shape)
print('X_validation.shape:', X_validation.shape)
print('X_test.shape:', X_test.shape)
print('X feature names:', FEATURE_COLUMNS)
print('X missing values:', int(pd.concat([X_train, X_validation, X_test]).isna().sum().sum()))
print('\nDATASET GENERATION COMPLETE')
print('Rows:', len(features))
print('Unique entities:', features.entity.nunique())
print('Unique timestamps:', features.timestamp.nunique())
print('Timestamp range:', int(features.timestamp.min()), '..', int(features.timestamp.max()))
print('Failed-auth rows:', int((features.failed_auth_count > 0).sum()))
print('Failed-auth events:', int(features.failed_auth_count.sum()))
print('Missing values:', missing_values)
print('Duplicate rows:', int(features.duplicated().sum()))
print('Training period:', int(train.timestamp.min()), '..', int(train.timestamp.max()))
print('Validation period:', int(validation.timestamp.min()), '..', int(validation.timestamp.max()))
print('Test period:', int(test.timestamp.min()), '..', int(test.timestamp.max()))
print('Constant features:', constant_features)
print('Highly redundant features:', highly_redundant)
status = 'READY FOR ISOLATION FOREST' if len(features) >= 50000 and features.entity.nunique() >= 500 else 'NEEDS MORE DATA'
print('DATASET STATUS:', status)
